In [28]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
import pandas as pd
import numpy as np

train_path = "/content/drive/MyDrive/Colab Notebooks/threat_6/UNSW_NB15_synthetic_train_250k.csv"

train_df = pd.read_csv(train_path)

print("Dataset shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

Dataset shape: (250000, 45)

Columns:
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']


In [30]:
print("Missing values:", train_df.isnull().sum().sum())
print("Duplicate rows:", train_df.duplicated().sum())

print("\nLabel distribution:")
print(train_df["label"].value_counts())

print("\nAttack categories:")
print(train_df["attack_cat"].value_counts())

Missing values: 0
Duplicate rows: 0

Label distribution:
label
1    159769
0     90231
Name: count, dtype: int64

Attack categories:
attack_cat
Normal            90231
Generic           57265
Exploits          43156
Fuzzers           23520
DoS               15818
Reconnaissance    13458
Analysis           2643
Backdoor           2290
Shellcode          1450
Worms               169
Name: count, dtype: int64


In [31]:
print("Data types:")
print(train_df.dtypes)

print("\nCategorical columns:")
print(train_df.select_dtypes(include="object").columns.tolist())

Data types:
id                     int64
dur                  float64
proto                 object
service               object
state                 object
spkts                float64
dpkts                float64
sbytes               float64
dbytes               float64
rate                 float64
sttl                 float64
dttl                 float64
sload                float64
dload                float64
sloss                float64
dloss                float64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                 float64
stcpb                float64
dtcpb                float64
dwin                 float64
tcprtt               float64
synack               float64
ackdat               float64
smean                float64
dmean                float64
trans_depth          float64
response_body_len    float64
ct_srv_src           float64
ct_state_ttl         float64
ct_dst_ltm           float64
ct

In [32]:
DROP_COLUMNS = ["id", "attack_cat", "label"]

X_train = train_df.drop(columns=DROP_COLUMNS)
y_train = train_df["label"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (250000, 42)
y_train shape: (250000,)


In [33]:
categorical_features = ["proto", "service", "state"]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Categorical:", categorical_features)
print("Numerical features:", len(numerical_features))
print("Total features:", len(X_train.columns))

Categorical: ['proto', 'service', 'state']
Numerical features: 39
Total features: 42


In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [35]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", rf_model)
    ]
)

print("Model pipeline created.")

Model pipeline created.


In [36]:
print("Training Random Forest...")

rf_pipeline.fit(X_train, y_train)

print("Training completed.")

Training Random Forest...
Training completed.


In [37]:
test_path = "/content/drive/MyDrive/Colab Notebooks/threat_6/UNSW_NB15_synthetic_test_250k.csv"

test_df = pd.read_csv(test_path)

print("Test dataset shape:", test_df.shape)
print("Columns:", len(test_df.columns))

Test dataset shape: (250000, 45)
Columns: 45


In [38]:
X_test = test_df.drop(columns=["id", "attack_cat", "label"])
y_test = test_df["label"]

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (250000, 42)
y_test shape: (250000,)


In [39]:
print("Generating predictions...")

y_pred = rf_pipeline.predict(X_test)
y_prob = rf_pipeline.predict_proba(X_test)[:, 1]

print("Predictions completed.")

Generating predictions...
Predictions completed.


In [40]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("=" * 60)
print("UNSW-NB15 SYNTHETIC 250K TEST RESULTS")
print("=" * 60)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Attack"]
))

UNSW-NB15 SYNTHETIC 250K TEST RESULTS
Accuracy : 0.9470
Precision: 0.9539
Recall   : 0.9637
F1 Score : 0.9587
ROC-AUC  : 0.9918

Confusion Matrix:
[[ 82786   7445]
 [  5807 153962]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.93      0.92      0.93     90231
      Attack       0.95      0.96      0.96    159769

    accuracy                           0.95    250000
   macro avg       0.94      0.94      0.94    250000
weighted avg       0.95      0.95      0.95    250000



Now, let's look at the feature importances from the Random Forest model to understand which features contributed most to the classification.

In [44]:
# Get feature importances from the Random Forest classifier
feature_importances = rf_pipeline.named_steps['classifier'].feature_importances_

# Get feature names from the preprocessor
onehot_features = rf_pipeline.named_steps['preprocessor'].named_transformers_['categorical'].get_feature_names_out(categorical_features)
all_feature_names = list(onehot_features) + numerical_features

# Create a DataFrame for feature importances
feature_importance_df = pd.DataFrame({
    'feature': all_feature_names,
    'importance': feature_importances
})

# Sort by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)

print("Top 10 Feature Importances:")
display(feature_importance_df.head(10))

Top 10 Feature Importances:


,feature,importance
162,sttl,0.097454
184,ct_state_ttl,0.072023
163,dttl,0.048674
164,sload,0.046807
179,smean,0.043828
188,ct_dst_src_ltm,0.038696
193,ct_srv_dst,0.037677
180,dmean,0.036452
187,ct_dst_sport_ltm,0.034535
165,dload,0.034177


In [41]:
import joblib

model_path = "/content/drive/MyDrive/Colab Notebooks/threat_6/rf_network_attack_detector.pkl"

joblib.dump(rf_pipeline, model_path)

print("Model saved successfully!")
print(model_path)

Model saved successfully!
/content/drive/MyDrive/Colab Notebooks/threat_6/rf_network_attack_detector.pkl


Let's save the model again, but this time using compression to reduce the `.pkl` file size. A compression level of `3` is a good balance between file size reduction and saving/loading speed.

In [43]:
import joblib

# Save the model with compression
compressed_model_path = "/content/drive/MyDrive/Colab Notebooks/threat_6/rf_network_attack_detector_compressed.pkl"
joblib.dump(rf_pipeline, compressed_model_path, compress=3)

print("Compressed model saved successfully!")
print(compressed_model_path)

Compressed model saved successfully!
/content/drive/MyDrive/Colab Notebooks/threat_6/rf_network_attack_detector_compressed.pkl


You can check the size of this new file to see the reduction. If further reduction is needed, you could consider reducing the `n_estimators` or adding a `max_depth` to the `RandomForestClassifier` (which would require re-training the model).

In [42]:
import joblib

loaded_model = joblib.load(
    "/content/drive/MyDrive/Colab Notebooks/threat_6/rf_network_attack_detector.pkl"
)

print("Model loaded successfully!")
print(type(loaded_model))

Model loaded successfully!
<class 'sklearn.pipeline.Pipeline'>
